# Clase 3 — Few-shot y zero-shot prompting

Una de las formas más efectivas de mejorar la calidad de una respuesta es **mostrarle al modelo ejemplos** de lo que esperás. Eso es few-shot prompting. En esta clase vamos a entender cuándo usarlo, cómo construir buenos ejemplos y qué diferencia hace la cantidad de ejemplos.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Zero-shot: el punto de partida |
| 3 | One-shot: un ejemplo cambia todo |
| 4 | Few-shot: cuántos ejemplos son suficientes |
| 5 | Cómo elegir buenos ejemplos |
| 6 | Actividad: diseñar una estrategia few-shot |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**

1. Obtené tu API key gratuita en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en un archivo `.env` en la carpeta del proyecto:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si preferís no crear el archivo, la celda siguiente te pide la clave de forma interactiva.

Si ya configuraste el entorno en clases anteriores, solo verificá que `BACKEND` esté correcto y ejecutá las celdas de setup.

In [8]:
# Seleciona aqui si usaras el modelo de Gemini o el modelo localmente
# Usa "gemini" para activar el backend de Gemini
# Usa "local" para usar un modelo local
BACKEND = "local"

In [9]:
import os
import getpass

# Elegir modelo de Gemini
GEMINI_MODEL = "gemma-4-26b-a4b-it"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

Backend: local


In [10]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

c:\Users\gonza\proyectos\proyectosia\clases-mainPromting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Entendido.


---
## 2. Zero-shot: el punto de partida

**Zero-shot** significa darle al modelo una instrucción sin ningún ejemplo. Es la forma más simple de prompting y funciona bien para tareas genéricas. El problema aparece cuando la tarea tiene un formato específico o un estilo particular que el modelo no puede adivinar.

| Modo | Ejemplos en el prompt | Cuándo usar |
|---|---|---|
| **Zero-shot** | 0 | Tareas genéricas, primeros intentos, cuando no tenés ejemplos buenos |
| **One-shot** | 1 | Cuando tenés un ejemplo claro y representativo |
| **Few-shot** | 2–5 | Cuando necesitás consistencia de formato o estilo específico |

In [11]:
# ─── Tarea de referencia ─────────────────────────────────────────────────────
# Vamos a clasificar el sentimiento de comentarios de clientes.
# Queremos: una sola palabra (Positivo / Negativo / Neutro) + una frase de justificación.

comentarios_prueba = [
    "El producto llegó antes de lo esperado y funciona perfecto.",
    "Tardó tres semanas y el empaque estaba roto.",
    "Lo usé una vez, sirve para lo que dice."
]

# ─── Zero-shot ────────────────────────────────────────────────────────────────
def clasificar_zero_shot(comentario):
    prompt = f"Clasificá el sentimiento de este comentario de cliente: '{comentario}'"
    return llamar_llm(prompt, max_tokens=60)


print("=== ZERO-SHOT ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_zero_shot(c)}")
    print()

=== ZERO-SHOT ===
Comentario: El producto llegó antes de lo esperado y funciona perfecto.
Respuesta:  Positivo.

Comentario: Tardó tres semanas y el empaque estaba roto.
Respuesta:  Negativo.

Comentario: Lo usé una vez, sirve para lo que dice.
Respuesta:  Negativo. El cliente expresa insatisfacción con la utilidad del producto.



> 💡 **Observá:** con zero-shot el modelo entiende la tarea, pero el formato de la respuesta varía — a veces da una palabra, a veces un párrafo. Eso dificulta procesar los resultados automáticamente.

---
## 3. One-shot: un ejemplo cambia todo

Con un solo ejemplo bien elegido, el modelo entiende no solo *qué* hacer sino *cómo* quiero que lo presente.

In [12]:
# ─── One-shot ─────────────────────────────────────────────────────────────────
# Ahora incluimos un ejemplo explícito de cómo debe verse la respuesta.

def clasificar_one_shot(comentario):
    prompt = """Clasificá el sentimiento de comentarios de clientes.
El formato de respuesta es: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase corta>

Ejemplo:
Comentario: "El servicio fue rápido pero el producto tenía un defecto menor."
Sentimiento: Neutro | Razón: mezcla aspectos positivos y negativos sin dominante claro.

Ahora clasificá este:
Comentario: "{comentario}"""

    return llamar_llm(prompt.format(comentario=comentario), max_tokens=60)


print("=== ONE-SHOT ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_one_shot(c)}")
    print()

=== ONE-SHOT ===
Comentario: El producto llegó antes de lo esperado y funciona perfecto.
Respuesta:  Sentimiento: Positivo | Razón: El producto llegó a tiempo y funciona correctamente.

Comentario: Tardó tres semanas y el empaque estaba roto.
Respuesta:  Sentimiento: Negativo | Razón: El retraso en la entrega y el empaque dañado indican una experiencia negativa.

Comentario: Lo usé una vez, sirve para lo que dice.
Respuesta:  Sentimiento: Neutro | Razón: Describe una experiencia básica con un producto sin expresar una opinión general.



---
## 4. Few-shot: cuántos ejemplos son suficientes

Agregar más ejemplos generalmente mejora la consistencia, pero llega un punto de rendimientos decrecientes. Después de 3–5 ejemplos bien elegidos, agregar más raramente cambia el resultado.

In [13]:
# ─── Few-shot con 3 ejemplos ──────────────────────────────────────────────────

ejemplos_few_shot = [
    ("La atención fue excelente y me solucionaron el problema en minutos.",
     "Sentimiento: Positivo | Razón: resolución rápida y efectiva."),
    ("El producto no coincide con las fotos del sitio.",
     "Sentimiento: Negativo | Razón: discrepancia entre lo mostrado y lo recibido."),
    ("Funciona como indica el manual, sin más ni menos.",
     "Sentimiento: Neutro | Razón: cumple expectativas básicas sin destacar."),
]

def clasificar_few_shot(comentario, ejemplos):
    # Construimos el bloque de ejemplos dinámicamente
    bloque_ejemplos = "\n".join(
        f'Comentario: "{inp}"\n{out}'
        for inp, out in ejemplos
    )

    prompt = f"""Clasificá el sentimiento de comentarios de clientes.
    Formato: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase corta>

    {bloque_ejemplos}

    Ahora clasificá este:
    Comentario: "{comentario}"""

    return llamar_llm(prompt, max_tokens=60)


print("=== FEW-SHOT (3 ejemplos) ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_few_shot)}")
    print()

=== FEW-SHOT (3 ejemplos) ===
Comentario: El producto llegó antes de lo esperado y funciona perfecto.
Respuesta:  Sentimiento: Positivo | Razón: Entrega rápida y funcionamiento óptimo.

Comentario: Tardó tres semanas y el empaque estaba roto.
Respuesta:  Sentimiento: Negativo | Razón: retraso en la entrega y daño al empaque.

Comentario: Lo usé una vez, sirve para lo que dice.
Respuesta:  Sentimiento: Neutro | Razón: Describe la utilidad básica del producto.



In [14]:
# ─── Comparación de consistencia por cantidad de ejemplos ─────────────────────
# Ejecutamos el mismo comentario con 0, 1 y 3 ejemplos y comparamos.

comentario_test = "El envío tardó más de lo prometido pero el producto es de buena calidad."

print("Comentario:", comentario_test)
print()
print("Zero-shot: ", clasificar_zero_shot(comentario_test))
print("One-shot:  ", clasificar_one_shot(comentario_test))
print("Few-shot:  ", clasificar_few_shot(comentario_test, ejemplos_few_shot))

Comentario: El envío tardó más de lo prometido pero el producto es de buena calidad.

Zero-shot:  Positivo.
One-shot:   Sentimiento: Positivo | Razón: El envío fue rápido y el producto es de buena calidad.
Few-shot:   Sentimiento: Neutro | Razón: El envío es tardado, pero la calidad del producto es buena.


---
## 5. Cómo elegir buenos ejemplos

No todos los ejemplos son igual de útiles. Estos criterios ayudan a elegirlos:

| Criterio | Qué significa |
|---|---|
| **Representatividad** | El ejemplo cubre un caso típico, no un caso raro |
| **Diversidad** | Si tenés varios ejemplos, que no sean todos del mismo tipo |
| **Claridad** | El par input-output es inequívoco; no hay ambigüedad en por qué la respuesta es esa |
| **Longitud similar** | Los ejemplos deberían tener longitud parecida al input real |
| **Sin contradicciones** | Dos ejemplos no deben sugerir respuestas distintas para inputs similares |

_
> 💡 Un ejemplo mal elegido puede confundir más que ayudar. Antes de agregar ejemplos, verificá que cada uno cumpla estos criterios.

In [16]:
# ─── Efecto de un ejemplo de mala calidad ─────────────────────────────────────
# Incluimos un ejemplo contradictorio para ver cómo afecta al modelo.

ejemplos_malos = [
    # Ejemplo contradictorio: input claramente positivo, output incorrecto
    ("Excelente atención, quedé muy satisfecho.",
     "Sentimiento: Negativo | Razón: el cliente mencionó un problema."),
]

print("=== FEW-SHOT CON EJEMPLO CONTRADICTORIO ===")
for c in comentarios_prueba[:2]:   # solo los primeros dos para ahorrar tokens
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_malos)}")
    print()

print("Comparar con few-shot limpio:")
for c in comentarios_prueba[:2]:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_few_shot)}")
    print()

=== FEW-SHOT CON EJEMPLO CONTRADICTORIO ===
Comentario: El producto llegó antes de lo esperado y funciona perfecto.
Respuesta:  Sentimiento: Positivo | Razón: El cliente elogió la rapidez de entrega y el buen funcionamiento del producto.

Comentario: Tardó tres semanas y el empaque estaba roto.
Respuesta:  Sentimiento: Negativo | Razón: El cliente expresa insatisfacción con el tiempo de entrega y el estado del empaque.

Comparar con few-shot limpio:
Comentario: El producto llegó antes de lo esperado y funciona perfecto.
Respuesta:  Sentimiento: Positivo | Razón: Entrega puntual y funcionamiento óptimo.

Comentario: Tardó tres semanas y el empaque estaba roto.
Respuesta:  Sentimiento: Negativo | Razón: retraso significativo en la entrega y daño al empaque.



---
## 6. Actividad: diseñar y probar estrategias few-shot

Vamos a hacer dos ejercicios: uno individual y uno grupal. En ambos casos, compará **zero-shot** vs **few-shot** y observá qué cambia en formato, consistencia, errores y manejo de casos ambiguos.

In [19]:
# Preparamos funciones para comparar zero-shot vs few-shot de forma más sistemática.

def construir_prompt_tarea(instruccion, formato, ejemplos, entrada):
    bloque_ejemplos = "\n\n".join(
        f"Entrada: {entrada_ejemplo}\nSalida: {salida_esperada}"
        for entrada_ejemplo, salida_esperada in ejemplos
    )

    return f"""{instruccion}

Formato de salida:
{formato}

Ejemplos:
{bloque_ejemplos}

Ahora resolvé:
Entrada: {entrada}
Salida:"""


def comparar_zero_vs_few(instruccion, formato, ejemplos, inputs_prueba, max_tokens=120):
    for entrada in inputs_prueba:
        prompt_zero = f"""{instruccion}

Formato de salida:
{formato}

Entrada: {entrada}
Salida:"""
        prompt_few = construir_prompt_tarea(instruccion, formato, ejemplos, entrada)

        print("Input:", entrada)
        print("Zero-shot:", llamar_llm(prompt_zero, max_tokens=max_tokens))
        print("Few-shot: ", llamar_llm(prompt_few, max_tokens=max_tokens))
        print("-" * 80)

print("Funciones listas")

Funciones listas


### Ejercicio grupal: cuando la guía falla

En grupos, diseñen un prompt para clasificar mensajes de soporte en una de las siguientes categorías:

| Categoría | Cuándo usarla |
|---|---|
| `Facturación` | Pagos, facturas, cobros, reembolsos |
| `Acceso` | Login, contraseña, 2FA, cuenta bloqueada |
| `Bug` | Algo que funcionaba dejó de funcionar o muestra un error |
| `Consulta` | Pregunta general sin incidente claro |
| `No clasificable` | Falta información, hay varias categorías empatadas o el mensaje no permite decidir |

Como vamos a usar un modelo chico, Gemma 3 de 1B, los casos están diseñados para que una guía simple falle. El objetivo no es que el primer prompt acierte todo, sino detectar errores y refinar la regla.

La parte desafiante: varios mensajes mezclan señales y obligan a decidir qué pesa más:

- mensajes con dos problemas a la vez;
- lenguaje emocional sin dato concreto;
- pedido implícito, sarcasmo o ironía;
- información insuficiente;
- palabras gatillo engañosas, por ejemplo mencionar "pago" dentro de un problema de acceso;
- acciones de facturación que fallan técnicamente, por ejemplo un botón o una pantalla que no carga.

Casos de prueba sugeridos:

- "Ayer no pude entrar; hoy entré y veo dos facturas iguales. ¿Esto lo revisan ustedes o tengo que cerrar sesión?"
- "Cuando toco Pagar, la pantalla queda pensando y después no guarda el cambio de plan; igual el banco me mandó aviso de consumo."
- "¿La factura de abril se descarga desde Perfil o tengo que activar algo de la cuenta primero?"
- "Mi cuenta está rara: a veces entra, a veces no, y no sé si es por la tarjeta nueva."
- "Error 502 al cambiar la contraseña; después me llegó un mail de pago pendiente."
- "Estoy cambiando la contraseña y me aparece 'pago rechazado'; no intenté comprar nada."
- "El botón Descargar factura no responde, pero solo me pasa después de iniciar sesión con Google."
- "Si cambio la contraseña, ¿se corta la suscripción que paga mi empresa?"
- "Buenísimo: primero me bloquearon, después me cobraron doble y ahora el chat dice que todo está perfecto."
- "La factura se abre en blanco; el resto del portal carga bien y puedo pagar sin problema."
- "Me llegó un mail que dice 'pagá hoy o bloqueamos la cuenta'; no sé si es real ni dónde verlo."
- "Quiero cambiar el mail de facturación, pero no puedo confirmar el código porque llega a la casilla vieja."

_
```
======================================================================

Entregable del grupo:

1. Una regla de decisión en 4 a 6 líneas.
2. Tres ejemplos few-shot bien elegidos.
3. Dos casos donde el primer prompt falló y cómo lo ajustaron.
4. Una comparación breve entre zero-shot y few-shot.

======================================================================
```

In [30]:
# ─── Ejercicio grupal ─────────────────────────────────────────────────────────
# Punto de partida: cada grupo debería ajustar estas reglas después de ver fallas.

# instruccion_grupal = """Clasificá mensajes de soporte en una sola categoría.
# Reglas de decisión:
#- Si el problema principal es un pago, factura, cobro o reembolso, usá Facturación.
#- Si el problema principal es entrar a la cuenta, contraseña, 2FA o bloqueo, usá Acceso.
#- Si una acción falla por un error técnico concreto, usá Bug.
#- Si es una pregunta general sin incidente claro, usá Consulta.
#- Si faltan datos o hay dos problemas igual de importantes, usá No clasificable."""'''

instruccion_grupal = """Clasifica mensajes de soporte en una sola categoría.
Reglas de decisión:
1. Si el mensaje menciona un ERROR TÉCNICO concreto (código de error, pantalla en blanco, botón que no responde), usá Bug.
2. Si el usuario NO PUEDE autenticarse, entrar, o recuperar acceso a su cuenta, usá Acceso.
3. Si el RESULTADO final es un problema económico (cobro duplicado, cargo incorrecto, falta reembolso), usá Facturación.
4. Si SOLO pregunta cómo hacer algo sin reportar falla o incidente, usá Consulta.
5. Si hay DOS problemas igual de importantes O falta información crítica para decidir, usá No clasificable."""

formato_grupal = "Categoría: <Facturación|Acceso|Bug|Consulta|No clasificable> | Razón: <explica con una frase corta>"

ejemplos_grupales = [
    (
        "No puedo entrar a mi cuenta desde ayer; el código 2FA nunca llega.",
        "Categoría: Acceso | Razón: el problema principal es autenticación."
    ),
    (
        "Me cobraron dos veces la suscripción de mayo.",
        "Categoría: Facturación | Razón: reporta un cobro duplicado."
    ),
    (
        "No puedo entrar y además creo que me cobraron mal.",
        "Categoría: No clasificable | Razón: mezcla dos problemas importantes sin prioridad clara."
    ),
    (
        "La pantalla  se abre en blanco.",
        "Categoría: Bug | Razón: pantalla en blanco es error técnico de visualización."
    ),
    (
        "Me bloquearon la cuenta y me cobraron igual.",
        "Categoría: Facturación | Razón: el problema económico (cobro indebido) es el resultado final."
    ),
    (
        "HOY ENTRE Y VEO 3 FACTURAS IGUALES Y NO SE SI ME COBRARON 3 VECES O ES UN ERROR DE VISUALIZACIÓN?",
        "Categoría: FACTURACIÓN | Razón: Se trata de un problema de cobros"
    ),
]

casos_grupales = [
    "Ayer no pude entrar; hoy entré y veo dos facturas iguales. ¿Esto lo revisan ustedes o tengo que cerrar sesión?",
    "Cuando toco Pagar, la pantalla queda pensando y después no guarda el cambio de plan; igual el banco me mandó aviso de consumo.",
    "¿La factura de abril se descarga desde Perfil o tengo que activar algo de la cuenta primero?",
    "Mi cuenta está rara: a veces entra, a veces no, y no sé si es por la tarjeta nueva.",
    "Error 502 al cambiar la contraseña; después me llegó un mail de pago pendiente.",
    "Estoy cambiando la contraseña y me aparece 'pago rechazado'; no intenté comprar nada.",
    "El botón Descargar factura no responde, pero solo me pasa después de iniciar sesión con Google.",
    "Si cambio la contraseña, ¿se corta la suscripción que paga mi empresa?",
    "Buenísimo: primero me bloquearon, después me cobraron doble y ahora el chat dice que todo está perfecto.",
    "La factura se abre en blanco; el resto del portal carga bien y puedo pagar sin problema.",
    "Me llegó un mail que dice 'pagá hoy o bloqueamos la cuenta'; no sé si es real ni dónde verlo.",
    "Quiero cambiar el mail de facturación, pero no puedo confirmar el código porque llega a la casilla vieja.",
]

# Descomentá para probar:
comparar_zero_vs_few(instruccion_grupal, formato_grupal, ejemplos_grupales, casos_grupales)

Input: Ayer no pude entrar; hoy entré y veo dos facturas iguales. ¿Esto lo revisan ustedes o tengo que cerrar sesión?
Zero-shot: Categoría: Consulta | Razón: El usuario pregunta cómo hacer algo sin reportar una falla o incidente.
Few-shot:  Categoría: No clasificable | Razón: Se trata de un problema de acceso y no hay información sobre si se revisa o no.
--------------------------------------------------------------------------------
Input: Cuando toco Pagar, la pantalla queda pensando y después no guarda el cambio de plan; igual el banco me mandó aviso de consumo.
Zero-shot: Categoría: Facturación | Razón: El usuario experimentó un error al realizar un pago, resultando en una falta de guardado del cambio de plan y un aviso de consumo.
Few-shot:  Categoría: Facturación | Razón: Falta de guardado de cambio de plan y notificación de consumo.
--------------------------------------------------------------------------------
Input: ¿La factura de abril se descarga desde Perfil o tengo que ac